# Robust SARIMAX Baseline for Hybrid AQI Pipeline

This notebook upgrades the SARIMA baseline to an optimized SARIMAX module with auto-ARIMA order search, stationarity-aware target transformation, exogenous weather regressors, and annual Fourier terms.

## 1) Imports

In [5]:
import warnings
warnings.filterwarnings('ignore')

import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import boxcox, skew
from scipy.special import inv_boxcox

from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_squared_error, r2_score

try:
    import pmdarima as pm
except ModuleNotFoundError:
    print('pmdarima not found. Installing...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pmdarima'])
    import pmdarima as pm

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Imports ready.')

Imports ready.


## 2) Load Data and Basic Preparation

In [6]:
df_raw = pd.read_csv('Dataset.csv')
df_raw = df_raw.rename(columns={'City': 'city', 'Date': 'date'})
df_raw['date'] = pd.to_datetime(df_raw['date'])

TARGET = 'AQI'
aqi_counts = df_raw.groupby('city')[TARGET].apply(lambda s: s.notna().sum()).sort_values(ascending=False)

CITY = 'Ahmedabad'
if CITY not in aqi_counts.index or aqi_counts.get(CITY, 0) == 0:
    CITY = aqi_counts.index[0]

df = df_raw[df_raw['city'] == CITY].copy().sort_values('date').set_index('date')

POLLUTANTS = [c for c in ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3'] if c in df.columns]
WEATHER_CANDIDATES = [c for c in ['t2m', 'd2m', 'sp', 'blh', 'u10', 'v10'] if c in df.columns]

keep_cols = POLLUTANTS + WEATHER_CANDIDATES + [TARGET]
df = df[keep_cols].copy()

for c in df.columns:
    df[c] = pd.to_numeric(df[c], errors='coerce')

feature_cols = [c for c in df.columns if c != TARGET]
df[feature_cols] = df[feature_cols].ffill().bfill().fillna(0)
df = df.dropna(subset=[TARGET]).copy()

df = df.asfreq('D')
df[feature_cols] = df[feature_cols].ffill().bfill().fillna(0)
df = df.dropna(subset=[TARGET]).copy()

print('City:', CITY)
print('Rows:', len(df))
print('Date range:', df.index.min(), 'to', df.index.max())

City: Ahmedabad
Rows: 1334
Date range: 2015-01-29 00:00:00 to 2020-07-01 00:00:00


## 3) Train/Test Split (Temporal 80/20)

In [7]:
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

y_train_raw = train_df[TARGET].copy()
y_test_raw = test_df[TARGET].copy()

print('Train rows:', len(train_df), '| Test rows:', len(test_df))
print('Train range:', train_df.index.min(), 'to', train_df.index.max())
print('Test range :', test_df.index.min(), 'to', test_df.index.max())

Train rows: 1067 | Test rows: 267
Train range: 2015-01-29 00:00:00 to 2019-09-30 00:00:00
Test range : 2019-10-01 00:00:00 to 2020-07-01 00:00:00


## 4) ADF Test + Automated Target Transformation (Log or Box-Cox)

In [8]:
def adf_report(series, name='series'):
    stat, pval, _, _, _, _ = adfuller(series.dropna(), autolag='AIC')
    print(f'ADF ({name}) statistic: {stat:.4f}, p-value: {pval:.6f}')
    return pval

def choose_transform(y):
    y = y.astype(float).copy()
    shift = 0.0
    if y.min() <= 0:
        shift = 1.0 - y.min()
    y_pos = y + shift

    y_log = np.log(y_pos)
    y_box, lam = boxcox(y_pos)

    score_log = abs(skew(y_log))
    score_box = abs(skew(y_box))

    if score_log <= score_box:
        method = 'log'
        y_trans = pd.Series(y_log, index=y.index)
        params = {'shift': shift}
    else:
        method = 'boxcox'
        y_trans = pd.Series(y_box, index=y.index)
        params = {'shift': shift, 'lambda': lam}

    return method, y_trans, params

def inverse_transform(y_trans, method, params):
    y_trans = np.asarray(y_trans)
    shift = params.get('shift', 0.0)

    if method == 'log':
        y_inv = np.exp(y_trans) - shift
    elif method == 'boxcox':
        lam = params['lambda']
        y_inv = inv_boxcox(y_trans, lam) - shift
    else:
        y_inv = y_trans

    return y_inv

print('ADF on raw train target:')
raw_p = adf_report(y_train_raw, 'AQI_raw_train')

transform_method, y_train_trans, transform_params = choose_transform(y_train_raw)
print('Chosen transform:', transform_method, '| params:', transform_params)

print('ADF on transformed train target:')
trans_p = adf_report(y_train_trans, 'AQI_trans_train')

ADF on raw train target:
ADF (AQI_raw_train) statistic: -3.9170, p-value: 0.001914
Chosen transform: boxcox | params: {'shift': 0.0, 'lambda': np.float64(0.009253412668449687)}
ADF on transformed train target:
ADF (AQI_trans_train) statistic: -3.7103, p-value: 0.003971


## 5) Build Exogenous Matrix: Top Correlated Weather + Annual Fourier (s=365)

In [9]:
# Select most strongly linearly correlated weather features using train split only
weather_corr = train_df[WEATHER_CANDIDATES + [TARGET]].corr(numeric_only=True)[TARGET].drop(TARGET).abs().sort_values(ascending=False)
top_k = min(3, len(weather_corr))
top_weather = list(weather_corr.head(top_k).index)

print('Top correlated weather features:', top_weather)
print(weather_corr.head(top_k))

# Fourier terms for annual seasonality (s=365)
def make_fourier(index, period=365, K=3):
    t = np.arange(len(index), dtype=float)
    out = pd.DataFrame(index=index)
    for k in range(1, K + 1):
        out[f'fourier_sin_{period}_{k}'] = np.sin(2 * np.pi * k * t / period)
        out[f'fourier_cos_{period}_{k}'] = np.cos(2 * np.pi * k * t / period)
    return out

K_ANNUAL = 3
fourier_train = make_fourier(train_df.index, period=365, K=K_ANNUAL)
fourier_test = make_fourier(test_df.index, period=365, K=K_ANNUAL)

exog_train = pd.concat([train_df[top_weather], fourier_train], axis=1)
exog_test = pd.concat([test_df[top_weather], fourier_test], axis=1)

exog_train = exog_train.ffill().bfill().fillna(0)
exog_test = exog_test.ffill().bfill().fillna(0)

print('Exog train shape:', exog_train.shape)
print('Exog test shape :', exog_test.shape)
display(exog_train.head())

Top correlated weather features: []
Series([], Name: AQI, dtype: float64)
Exog train shape: (1067, 6)
Exog test shape : (267, 6)


,fourier_sin_365_1,fourier_cos_365_1,fourier_sin_365_2,fourier_cos_365_2,fourier_sin_365_3,fourier_cos_365_3
date,,,,,,
2015-01-29,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000
2015-01-30,0.017213,0.999852,0.034422,0.999407,0.051620,0.998667
2015-01-31,0.034422,0.999407,0.068802,0.997630,0.103102,0.994671
2015-02-01,0.051620,0.998667,0.103102,0.994671,0.154309,0.988023
2015-02-02,0.068802,0.997630,0.137279,0.990532,0.205104,0.978740


## 6) Hyperparameter Optimization with Auto-ARIMA (m=7) and SARIMAX Fit

In [10]:
auto_model = pm.auto_arima(
    y=y_train_trans,
    X=exog_train,
    seasonal=True,
    m=7,
    start_p=0, start_q=0,
    max_p=3, max_q=3,
    start_P=0, start_Q=0,
    max_P=2, max_Q=2,
    d=None, D=None,
    test='adf',
    seasonal_test='ocsb',
    information_criterion='aic',
    stepwise=False,
    n_jobs=-1,
    suppress_warnings=True,
    error_action='ignore',
    trace=True
)

print('Best non-seasonal order:', auto_model.order)
print('Best seasonal order   :', auto_model.seasonal_order)
print('Best AIC              :', auto_model.aic())

# 5) Output generation for hybrid stage
# In-sample predictions on transformed scale, then inverse transform
train_pred_trans = auto_model.predict_in_sample(X=exog_train)
test_pred_trans = auto_model.predict(n_periods=len(test_df), X=exog_test)

sarimax_pred_train = pd.Series(inverse_transform(train_pred_trans, transform_method, transform_params), index=train_df.index)
sarimax_pred_test = pd.Series(inverse_transform(test_pred_trans, transform_method, transform_params), index=test_df.index)

# Align as full timeline so downstream XGBoost can fuse these features
sarimax_pred_full = pd.concat([sarimax_pred_train, sarimax_pred_test]).sort_index()

print('In-sample prediction length :', len(sarimax_pred_train))
print('Out-of-sample forecast length:', len(sarimax_pred_test))


Best model:  ARIMA(1,0,3)(0,0,0)[7] intercept
Total fit time: 29.656 seconds
Best non-seasonal order: (1, 0, 3)
Best seasonal order   : (0, 0, 0, 7)
Best AIC              : 1155.7111535976337
In-sample prediction length : 1067
Out-of-sample forecast length: 267


## 7) Evaluate Optimized SARIMAX Baseline

In [11]:
train_mse = mean_squared_error(y_train_raw, sarimax_pred_train)
train_r2 = r2_score(y_train_raw, sarimax_pred_train)

test_mse = mean_squared_error(y_test_raw, sarimax_pred_test)
test_r2 = r2_score(y_test_raw, sarimax_pred_test)

print('Optimized SARIMAX (with exog + annual Fourier)')
print(f'  Train MSE: {train_mse:.4f}')
print(f'  Train R2 : {train_r2:.4f}')
print(f'  Test MSE : {test_mse:.4f}')
print(f'  Test R2  : {test_r2:.4f}')

Optimized SARIMAX (with exog + annual Fourier)
  Train MSE: 51641.6365
  Train R2 : 0.4669
  Test MSE : 69056.8586
  Test R2  : 0.0692


## 8) Export Blocks for XGBoost Stage

In [12]:
# These are the core outputs you can merge into your hybrid XGBoost pipeline:
# - sarimax_pred_train: in-sample SARIMAX predictions for train segment
# - sarimax_pred_test : out-of-sample SARIMAX forecasts for test segment

train_df_out = train_df.copy()
test_df_out = test_df.copy()

train_df_out['SARIMAX_pred'] = sarimax_pred_train
test_df_out['SARIMAX_pred'] = sarimax_pred_test

print(train_df_out[['AQI', 'SARIMAX_pred']].head().to_string())
print('---')
print(test_df_out[['AQI', 'SARIMAX_pred']].head().to_string())

              AQI  SARIMAX_pred
date                           
2015-01-29  209.0    445.553382
2015-01-30  328.0    270.062665
2015-01-31  514.0    369.352625
2015-02-01  782.0    463.483187
2015-02-02  914.0    583.817790
---
              AQI  SARIMAX_pred
date                           
2019-10-01  266.0    456.022462
2019-10-02  430.0    512.212574
2019-10-03  528.0    536.094122
2019-10-04  512.0    538.693579
2019-10-05  614.0    541.243921
